# Marine microbial taxa ecological-group assignments and calculation template data from 16S rRNA metabarcoding studies Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In this notebook, we will:
- load the dataset metadata from the Croissant description,
- inspect available record sets and fields using their `@id` values,
- extract tabular data into pandas DataFrames,
- perform simple exploratory data analysis (EDA), and
- create a few visual summaries when tabular record sets are available.

> Important: all record sets, fields, and columns are referenced by their Croissant `@id` values whenever possible.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.rwz4-w7n2/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Access metadata as a single object
dataset_name = getattr(metadata, "name", None)
dataset_description = getattr(metadata, "description", None)
dataset_id = getattr(metadata, "id", None)

print("Dataset name:", dataset_name)
print("Dataset description:", dataset_description)
print("Dataset @id:", dataset_id)

In [ ]:
# Convert metadata to JSON-like form for inspection while keeping dataset.metadata itself as an object
metadata_json = metadata.to_json()

summary_fields = [
    "name",
    "description",
    "identifier",
    "version",
    "license",
    "datePublished",
    "spatialCoverage",
    "keywords",
]

summary = {k: metadata_json.get(k) for k in summary_fields}
pprint(summary)

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset package provided in the Croissant metadata suggests that `recordSet` is currently empty. We will verify that programmatically and build a robust workflow that still works if record sets are present or added in a future version.

In [ ]:
# Inspect the high-level Croissant metadata JSON for record sets and distributions
record_sets_json = metadata_json.get("recordSet", []) or []
distributions_json = metadata_json.get("distribution", []) or []

print(f"Number of record sets found: {len(record_sets_json)}")
print(f"Number of distributions found: {len(distributions_json)}")

if distributions_json:
    print("\nDistribution @id values:")
    for dist in distributions_json:
        if isinstance(dist, dict):
            print(" -", dist.get("@id"))

if not record_sets_json:
    print("\nNo record sets were declared in the Croissant metadata.")

In [ ]:
# Utility helpers to inspect record sets by @id if they exist
def collect_record_set_ids(record_sets):
    ids = []
    for rs in record_sets:
        if isinstance(rs, dict) and "@id" in rs:
            ids.append(rs["@id"])
    return ids

record_set_ids = collect_record_set_ids(record_sets_json)
print("record_set_ids:", record_set_ids)

for rs in record_sets_json:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    fields = rs.get("field", []) or []
    print("\nRecord set @id:", rs_id)
    print("Record set name:", rs_name)
    print("Number of fields:", len(fields))
    for field in fields:
        if isinstance(field, dict):
            print("   field @id:", field.get("@id"), "| name:", field.get("name"), "| dataType:", field.get("dataType"))

The template includes an example of iterating through records from a record set using its `@id`. The next cell demonstrates that pattern safely.

If no record sets exist, the cell will simply report that there is nothing to iterate over.

In [ ]:
# Example pattern from the template, adapted to work safely with this dataset
if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print("Inspecting records from record set @id:", sample_record_set_id)
    for i, x in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(x)
        if i >= 4:
            break
else:
    print("No record set @id is available, so dataset.records(record_set=<@id>) cannot be demonstrated on this version of the dataset.")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

Because this dataset currently exposes no record sets in its Croissant metadata, the extraction code below is written defensively. It will create DataFrames for any record sets if they become available, while still allowing us to analyze metadata-level information right now.

In [ ]:
# Extract data from each record set, keyed by record set @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} rows from record set @id: {record_set_id}")

if dataframes:
    first_record_set_id = next(iter(dataframes))
    print("\nColumns for first record set:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No tabular record-set data could be extracted because no record sets were declared.")

In [ ]:
# Build a metadata-level DataFrame from selected top-level dataset fields for lightweight analysis
top_level_items = []

for key, value in metadata_json.items():
    top_level_items.append({
        "field_key": key,
        "value_type": type(value).__name__,
        "is_list": isinstance(value, list),
        "is_dict": isinstance(value, dict),
        "string_length": len(value) if isinstance(value, str) else None,
        "list_length": len(value) if isinstance(value, list) else None,
        "preview": str(value)[:150]
    })

metadata_df = pd.DataFrame(top_level_items)
display(metadata_df.head(20))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Since there are no dataset record sets available in the current Croissant file, this section demonstrates EDA on metadata-derived tables and includes a reusable branch for record-set data if such tables are added later.

We will:
- inspect keyword information,
- analyze text lengths for selected metadata fields,
- categorize metadata fields by value type,
- and provide a generic numeric-field EDA workflow for any future record set.

In [ ]:
# Expand keywords into a tidy DataFrame
keywords = metadata_json.get("keywords", []) or []
keywords_df = pd.DataFrame({
    "keyword": keywords,
    "keyword_length": [len(k) for k in keywords]
})

print(f"Number of keywords: {len(keywords_df)}")
display(keywords_df.head(10))

In [ ]:
# Select a numeric field for analysis from the metadata-derived keyword table
numeric_field = "keyword_length"
threshold = 12

filtered_keywords_df = keywords_df[keywords_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_keywords_df.head())

if not filtered_keywords_df.empty and filtered_keywords_df[numeric_field].std() not in [0, None]:
    filtered_keywords_df[f"{numeric_field}_normalized"] = (
        (filtered_keywords_df[numeric_field] - filtered_keywords_df[numeric_field].mean()) /
        filtered_keywords_df[numeric_field].std()
    )
else:
    filtered_keywords_df[f"{numeric_field}_normalized"] = 0.0

print(f"Normalized {numeric_field} for filtered records:")
display(filtered_keywords_df[[numeric_field, f"{numeric_field}_normalized"]].head())

group_field = "first_letter"
filtered_keywords_df[group_field] = filtered_keywords_df["keyword"].str[0].str.upper()

if group_field in filtered_keywords_df.columns:
    grouped_df = filtered_keywords_df.groupby(group_field, as_index=False)[numeric_field].mean()
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

In [ ]:
# Analyze lengths of selected descriptive text fields
text_fields = [
    "description",
    "dataCollection",
    "dataCollectionMissingData",
    "dataCollectionRawData",
    "dataManipulationProtocol",
    "dataSocialImpact",
]

text_length_rows = []
for field_id in text_fields:
    value = metadata_json.get(field_id)
    if isinstance(value, str):
        text_length_rows.append({
            "field_id": field_id,
            "text_length": len(value)
        })

text_lengths_df = pd.DataFrame(text_length_rows)
display(text_lengths_df)

In [ ]:
# Category counts for top-level metadata field value types
value_type_counts = (
    metadata_df.groupby("value_type", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
)

display(value_type_counts)

In [ ]:
# Generic EDA workflow for future record sets with numeric columns
if dataframes:
    for record_set_id, df in dataframes.items():
        numeric_columns = df.select_dtypes(include="number").columns.tolist()
        print(f"\nRecord set @id: {record_set_id}")
        print("Numeric columns:", numeric_columns)
        
        if numeric_columns:
            numeric_field = numeric_columns[0]
            threshold = df[numeric_field].median()
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Using numeric field @id/column: {numeric_field}")
            print(f"Threshold: {threshold}")
            display(filtered_df.head())
            
            if not filtered_df.empty and filtered_df[numeric_field].std() not in [0, None]:
                filtered_df[f"{numeric_field}_normalized"] = (
                    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
                    filtered_df[numeric_field].std()
                )
                display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print("No numeric columns available for this record set.")
else:
    print("No record-set DataFrames are available for generic tabular EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Because the current Croissant metadata does not expose record sets, the visualizations below focus on metadata-derived tables.

In [ ]:
# Visualization 1: keyword lengths
if not keywords_df.empty:
    plt.figure(figsize=(10, 5))
    sns.histplot(keywords_df["keyword_length"], bins=10, kde=True)
    plt.title("Distribution of Dataset Keyword Lengths")
    plt.xlabel("Keyword length")
    plt.ylabel("Count")
    plt.show()
else:
    print("No keywords available for plotting.")

In [ ]:
# Visualization 2: text length by metadata field
if not text_lengths_df.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=text_lengths_df, x="field_id", y="text_length", palette="viridis")
    plt.title("Length of Selected Textual Metadata Fields")
    plt.xlabel("Metadata field @id/key")
    plt.ylabel("Number of characters")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No textual metadata fields available for plotting.")

In [ ]:
# Visualization 3: count of top-level metadata field types
if not value_type_counts.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=value_type_counts, x="value_type", y="count", palette="magma")
    plt.title("Top-level Metadata Fields by Value Type")
    plt.xlabel("Python value type")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No metadata type counts available for plotting.")

## 6. Conclusion
This notebook demonstrated how to load and inspect a Croissant dataset with `mlcroissant` using a URL-based schema.

Key observations from this dataset version:
- The dataset metadata loads successfully and provides rich descriptive information.
- The Croissant file currently contains **no declared record sets** in the `recordSet` section.
- Even without tabular record sets, the metadata itself can be explored programmatically.
- The dataset includes useful descriptive fields such as keywords, collection protocols, bias statements, and use cases.

If a future version of the dataset adds record sets, the extraction and EDA code in this notebook is already structured to use their `@id` values directly and extend the analysis to row-level data.